# Fusion photo-z (no-mask) — train on Colab (logs to MLflow)

MLP + MDN(K=5) over the **67-d** fusion input `[3 tabular base preds | 64 CNN embedding]` — no absence mask, complete features only (`fusion_input_v4_nomask.parquet`). Every val objid is one complete row, so val == the 45k `val50k`; logs `val50k_sigma_MAD` (comparable to the CNN / tabular runs).

In [ ]:
!pip -q install mlflow

In [ ]:
# --- MLflow connect (paste token at the prompt; it is NOT stored in the notebook) ---
import os, getpass, mlflow, mlflow.tensorflow
os.environ["MLFLOW_TRACKING_URI"] = "https://146-148-10-86.sslip.io"
os.environ["MLFLOW_TRACKING_TOKEN"] = getpass.getpass("MLflow token: ")
mlflow.set_experiment("fusion")
mlflow.tensorflow.autolog(log_models=False, log_datasets=False, checkpoint=False)
print("MLflow ->", os.environ["MLFLOW_TRACKING_URI"], "| experiment: fusion")

In [ ]:
# --- get the precomputed fusion inputs from GCS ---
from google.colab import auth; auth.authenticate_user()
!gsutil -q cp gs://macrocosm-lewagon/data/fusion_input_v4_nomask.parquet /content/fusion_input_v4_nomask.parquet
import pandas as pd
df = pd.read_parquet("/content/fusion_input_v4_nomask.parquet")
print(df.shape); print(df["split"].value_counts().to_dict())

In [ ]:
# --- fusion model + MDN helpers (self-contained) ---
import numpy as np, tensorflow as tf
from tensorflow.keras import layers as L, Model, Input, regularizers

def mdn_nll(K):
    def loss(y_true, y_pred):
        pi, mu, sig = y_pred[:, :K], y_pred[:, K:2*K], y_pred[:, 2*K:]
        y = tf.expand_dims(y_true, 1)
        lc = (tf.math.log(pi+1e-8) - 0.5*tf.math.log(2*np.pi*sig**2+1e-8) - 0.5*((y-mu)/(sig+1e-8))**2)
        return tf.reduce_mean(-tf.reduce_logsumexp(lc, axis=1))
    return loss

def mdn_point(raw):                          # log1p(z)-space point estimate (mu of max-pi component)
    raw = np.asarray(raw)
    if raw.ndim == 2 and raw.shape[1] > 1 and raw.shape[1] % 3 == 0:
        K = raw.shape[1]//3; pi, mu = raw[:, :K], raw[:, K:2*K]
        return mu[np.arange(len(mu)), pi.argmax(1)]
    return raw.ravel()

def build_fusion(n_in, hidden=(128,128,64), mdn=5, l2=1e-4, drop=0.3):
    reg = regularizers.l2(l2)
    inp = Input((n_in,), name="fusion_in"); x = L.BatchNormalization()(inp)
    for i, h in enumerate(hidden):
        x = L.Dense(h, activation="relu", kernel_regularizer=reg, name=f"fc{i+1}")(x)
        x = L.BatchNormalization()(x); x = L.Dropout(drop)(x)
    pi  = L.Dense(mdn, activation="softmax", name="mdn_pi")(x)
    mu  = L.Dense(mdn, name="mdn_mu")(x)
    sig = L.Dense(mdn, activation="softplus", name="mdn_sigma")(x)   # softplus: no exp-overflow -> stable across inits
    return Model(inp, L.Concatenate(name="z")([pi, mu, sig]), name=f"fusion-nomask-mdn{mdn}")

def delta_z(zt, zp): zt, zp = np.asarray(zt,float), np.asarray(zp,float); return (zp-zt)/(1+zt)
def sigma_mad(zt, zp): d = delta_z(zt, zp); return float(1.4826*np.median(np.abs(d-np.median(d))))
def outlier_rate(zt, zp, thr=0.05): return float(np.mean(np.abs(delta_z(zt, zp)) > thr))

In [ ]:
# --- build train/val matrices (67-d = 3 base + 64 emb, no mask) ---
base_c = ["base_RF", "base_HGB", "base_MLP"]
emb_c  = sorted([c for c in df.columns if c.startswith("emb_")], key=lambda c: int(c[4:]))
feat = base_c + emb_c
def xy(name):
    d = df[df.split == name]
    return (d[feat].to_numpy("float32"), np.log1p(d["redshift"].to_numpy("float32")),
            d["redshift"].to_numpy("float64"))
Xtr, ytr, _   = xy("train")
Xva, yva, zva = xy("val")                 # all complete -> val == val50k
print("in-dim", len(feat), "| train", Xtr.shape, "| val (val50k)", Xva.shape)

In [ ]:
# --- train (MDN NLL; early-stop on val50k_sigma_MAD, keep best weights) ---
class SigmaMadCB(tf.keras.callbacks.Callback):
    def __init__(self, X, z): self.X, self.z = X, z
    def on_epoch_end(self, e, logs=None):
        zp = np.expm1(mdn_point(self.model.predict(self.X, verbose=0)))
        sm = sigma_mad(self.z, zp); logs = logs or {}; logs["val50k_sigma_MAD"] = sm
        try:
            if mlflow.active_run(): mlflow.log_metric("val50k_sigma_MAD", float(sm), step=e)
        except Exception: pass
        print(f"  epoch {e}: val50k sigma_MAD={sm:.4f}")

K = 5
model = build_fusion(len(feat), hidden=(128,128,64), mdn=K, drop=0.3)
model.compile(optimizer=tf.keras.optimizers.Adam(3e-4, clipnorm=1.0), loss=mdn_nll(K))
cbs = [SigmaMadCB(Xva, zva),
       tf.keras.callbacks.EarlyStopping("val50k_sigma_MAD", mode="min", patience=10, restore_best_weights=True),
       tf.keras.callbacks.ReduceLROnPlateau("val50k_sigma_MAD", mode="min", factor=0.5, patience=3, min_lr=1e-5)]
mlflow.start_run(run_name="fusion-v4-nomask")
mlflow.log_params(dict(mdn=K, hidden="128-128-64", lr=3e-4, batch=1024, drop=0.3,
                       inputs="3base+64emb (no mask)", n_train=len(Xtr), n_val=len(Xva)))
model.fit(Xtr, ytr, validation_data=(Xva, yva), epochs=80, batch_size=1024, callbacks=cbs)

In [ ]:
# --- final val50k metrics + save best (restore_best_weights already applied) ---
zp = np.expm1(mdn_point(model.predict(Xva, verbose=0)))
sm, ou = sigma_mad(zva, zp), outlier_rate(zva, zp)*100
print(f"val50k  n={len(zva)}  sigma_MAD={sm:.5f}  outlier={ou:.2f}%")
mlflow.log_metric("val50k_sigma_MAD", sm); mlflow.log_metric("val50k_outlier", ou/100)
model.save("/content/fusion_v4_nomask.keras")
mlflow.log_artifact("/content/fusion_v4_nomask.keras")
mlflow.end_run()
print("done — best-val50k model logged to MLflow (experiment 'fusion')")